# Spatial QC of the Slide-tags positions, mouse and human

A nucleus whose Slide-tags position cannot be trusted loses the position (set to NaN) and stays in every other analysis; no nucleus is removed. Only coordinates drive the decisions; cell types are not consulted. The steps, in order:

1. **unmapped**: no position from Slide-tags. Left as is.
2. **degenerate**: within each 10x lane, a position that more than one nucleus shares exactly, or whose mean distance to its 5 nearest positions is below a crowding floor. Runs first so that pile-ups cannot distort the statistics of the next step.
3. **far**: per timepoint, lanes pooled, the mean distance to the 5 nearest remaining positions; a position is blanked when log(1 + distance) exceeds the median by more than 5 median absolute deviations.
4. **boxed**: hand-drawn exclusion regions for stragglers that the distance criteria miss.

Kept nuclei then get a tissue section (their lane, split by a line where one puck carries two sections) and a rotated copy of their position (all angles are 0).

The settings are those of the QC of record, round 2 of 30 Oct 2026, whose labels and QC'd coordinates release v9 (`2026_10_30`) carries. Mouse goes through every step. Human goes through exact duplicates only, the QC of the human release of 2 Sept 2026; the section *Human: what the mouse criteria would remove* shows what the other criteria would do. The notebook is a port of `workflow/scripts/atlas/spatial_qc.py`, itself a port of `01b_spatial_qc_mouse.ipynb`, and runs on its own, outside Snakemake.

**Inputs**, one per species: an h5ad whose obs holds `x_um_dbscan` and `y_um_dbscan` (Slide-tags DBSCAN positions, µm), `sample` (10x lane) and `timepoint` (sample-sheet names), for the nuclei the QC covers. The round-2 inputs hold the nuclei kept by the round-2 atlas tables, 60,049 mouse and 69,222 human; release v9 keeps 58,237 and 67,921 of them after annotation.

**Outputs**, in `OUT_DIR`, never overwritten unless `FORCE = True`:
- `qc_spatial_{species}.h5ad`: the input plus `spatial_qc_reason`, `x_um_qc`, `y_um_qc`, `x_um_rot`, `y_um_rot` and `tissue_section`; raw positions unchanged, `obsm['spatial']` QC'd, `obsm['spatial_rot']` rotated, settings in `uns['spatial_qc']`.
- `spatial_qc_{species}.csv`: outcome counts per timepoint.

**Checks**, last section: labels and coordinates must equal those of the QC of record and of the release objects.

In [ ]:
# Paths are relative to REPO, the project folder that holds the inputs, unless absolute.
REPO = "/n/lab_storage/eddy_lab/users/nhilgert/xmens_final"
INPUTS = {
    "mouse": "data/interim/round2_2026_10_29/atlas_kept_coords_mouse.h5ad",
    "human": "data/interim/round2_2026_10_29/atlas_kept_coords_human.h5ad",
}
OUT_DIR = "results/spatial_qc/round2_2026_10_29"  # None writes nothing
FORCE = False  # overwrite existing outputs in OUT_DIR

# Results to check against; None skips that check.
RECORD = {  # the QC of record, round 2
    "mouse": "data/interim/round2_2026_10_29/qc_spatial_mouse.h5ad",
    "human": "data/interim/round2_2026_10_29/qc_spatial_human.h5ad",
}
RECORD_TALLY = {
    "mouse": "results/tables/spatial_qc_mouse_round2.csv",
    "human": "results/tables/spatial_qc_human_round2.csv",
}
RELEASE = {  # release v9, 2026_10_30
    "mouse": "data/deliverables/2026_10_30/Cevrim_XMens_All_Cells.h5ad",
    "human": "data/deliverables/2026_10_30/Cevrim_Human_All_Cells.h5ad",
}

In [ ]:
import json
import operator
from pathlib import Path

import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from anndata.io import read_elem
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from sklearn.neighbors import NearestNeighbors

## Settings

Distances are in µm on the raw positions, each puck in its own frame. Timepoints use the sample-sheet names of the inputs; release v9 renames four of them (`1_dpi_early` is `1.5_dpi_early`, and so on), and the plots use the release names.

In [ ]:
X, Y = "x_um_dbscan", "y_um_dbscan"
SAMPLE = "sample"  # 10x lane; the lanes of one timepoint hold nuclei from one puck
TIMEPOINT = "timepoint"

K = 5  # neighbors in the mean nearest-neighbor distance of both distance criteria
MAD_SCALE = 1.0  # plain MADs; 1.4826 would express n_mads in standard deviations

# Exclusion regions for the mouse: all bounds of a region must hold. Drawn by hand on the raw
# positions for stragglers that the distance criteria miss. The dense island in the center of the
# 3 dpi section is not boxed: its nuclei have the same cell types as the surrounding tissue.
MOUSE_EXCLUSIONS = [
    # Uninduced, both lanes: the minimal union of the earlier boxes (2 Sept 2026).
    {"timepoint": "uninduced", "y_gt": 5000},  # above both sections
    {"timepoint": "uninduced", "x_lt": 500},  # left margin
    {"timepoint": "uninduced", "x_lt": 3000, "y_lt": 1950},  # below left
    {"timepoint": "uninduced", "x_lt": 3400, "y_lt": 700},  # isolated pair below the right section
    {"timepoint": "uninduced", "x_gt": 5400, "y_gt": 3000},  # top-right stragglers
    {"timepoint": "1_dpi_early", "x_lt": 750},
    {"timepoint": "2_dpi", "y_gt": 7000},
    {"timepoint": "3_dpi", "y_gt": 9000},
    {"timepoint": "3_dpi", "y_lt": 1500, "x_gt": 6500},
    {"timepoint": "3_dpi", "x_gt": 9000, "y_gt": 8000},
]

# The uninduced puck carries two tissue sections, separated by this line (chosen 2 Sept 2026 to
# agree with a two-cluster k-means split of the uninduced positions).
MOUSE_SECTIONS = [
    {"timepoint": "uninduced", "line": {"slope": 1.0, "intercept": -500}, "above": "A", "below": "B"},
]

# Rigid rotation per timepoint, in degrees, about each lane's median kept position. All 0: the QC'd
# positions stay in the raw frame.
MOUSE_ROTATIONS = {
    "uninduced": 0.0,
    "1_dpi_early": 0.0,
    "1_dpi_late": 0.0,
    "2_dpi": 0.0,
    "3_dpi": 0.0,
    "4_dpi_bleeding": 0.0,
}

SETTINGS = {
    "mouse": {
        "floor_um": 10.0,  # crowding floor on the mean distance to the K nearest positions of a lane
        "max_at_coord": 1,  # more nuclei than this at one exact coordinate of a lane: all degenerate
        "n_mads": 5.0,  # far cut, per timepoint
        "exclusions": MOUSE_EXCLUSIONS,
        "sections": MOUSE_SECTIONS,
        "rotations": MOUSE_ROTATIONS,
    },
    # Human: exact duplicates only, the QC of the human release of 2 Sept 2026. A floor of 0 and an
    # infinite cut switch the other criteria off; one section per lane, no rotation.
    "human": {
        "floor_um": 0.0,
        "max_at_coord": 1,
        "n_mads": np.inf,
        "exclusions": [],
        "sections": [],
        "rotations": {},
    },
}

TIMEPOINT_ORDER = {
    "mouse": ["uninduced", "1_dpi_early", "1_dpi_late", "2_dpi", "3_dpi", "4_dpi_bleeding"],
    "human": ["secretory", "menstrual"],
}
TIMEPOINT_LABELS = {  # release v9 names
    "uninduced": "Uninduced",
    "1_dpi_early": "1.5 dpi, early",
    "1_dpi_late": "1.5 dpi, late",
    "2_dpi": "2.5 dpi",
    "3_dpi": "3.5 dpi",
    "4_dpi_bleeding": "4.5 dpi, bleeding",
    "secretory": "Secretory",
    "menstrual": "Menstrual",
}

ORDER = ["kept", "far", "degenerate", "boxed", "unmapped"]  # categories of spatial_qc_reason

In [ ]:
# Kept positions in grey so the eye goes to the three blanking steps, which take Okabe-Ito hues
# that stay distinct under the common color-vision deficiencies.
OUTCOME_COLORS = {"kept": "#B0B0B0", "degenerate": "#D55E00", "far": "#0072B2", "boxed": "#CC79A7"}
REGION_COLOR = "#CC79A7"
CUT_COLOR = "#0072B2"

plt.rcParams.update(
    {
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "figure.dpi": 110,
    }
)

## Load

In [ ]:
repo = Path(REPO)


def resolve(path: str | None) -> Path | None:
    """A path parameter, relative to the repository root unless absolute."""
    return None if path is None else repo / path


SPECIES = list(INPUTS)
adatas = {species: ad.read_h5ad(resolve(path)) for species, path in INPUTS.items()}
obs = {}
for species, adata in adatas.items():
    missing = [col for col in (X, Y, SAMPLE, TIMEPOINT) if col not in adata.obs]
    if missing:
        raise KeyError(f"{INPUTS[species]} has no obs columns {missing}")

    unknown = sorted(set(adata.obs[TIMEPOINT].astype(str)) - set(TIMEPOINT_ORDER[species]))
    if unknown:
        raise ValueError(f"{species}: timepoints missing from TIMEPOINT_ORDER: {unknown}")

    obs[species] = adata.obs
    print(f"{species}: {adata.n_obs:,} nuclei in {adata.obs[SAMPLE].nunique()} lanes")

## Distances

Both distance criteria use the mean distance from a position to its `K` nearest other positions.

In [ ]:
def knn_mean_dist(xy: np.ndarray, k: int = K) -> np.ndarray:
    """Mean distance (µm) from each position to its k nearest other positions."""
    # A group of k + 1 or fewer positions has no k neighbors; it scores 0.
    if len(xy) <= k + 1:
        return np.zeros(len(xy))

    dist, _ = NearestNeighbors(n_neighbors=k + 1).fit(xy).kneighbors(xy)
    return dist[:, 1:].mean(axis=1)


def far_cut(dist_um: np.ndarray, n_mads: float) -> float:
    """Distance (µm) above which a position is far: median + n_mads MADs of log1p(distance)."""
    log_dist = np.log1p(dist_um)
    median = np.median(log_dist)
    mad = np.median(np.abs(log_dist - median))
    return float(np.expm1(median + n_mads * MAD_SCALE * mad))


def counts_by_timepoint(species: str, flags: dict[str, np.ndarray]) -> pd.DataFrame:
    """Number of nuclei per timepoint for each boolean flag, in display order."""
    timepoints = obs[species][TIMEPOINT].astype(str)
    table = pd.DataFrame(flags, index=obs[species].index).groupby(timepoints).sum()
    return table.reindex(TIMEPOINT_ORDER[species]).rename(index=TIMEPOINT_LABELS)

## 1. Unmapped

Many nuclei never receive a position from Slide-tags: their spatial barcodes form no DBSCAN cluster, or more than one. They keep their expression data and simply have no position.

In [ ]:
reason = {}
is_mapped = {}
for species in SPECIES:
    meta = obs[species]
    is_mapped[species] = (meta[X].notna() & meta[Y].notna()).to_numpy()
    reason[species] = pd.Series("kept", index=meta.index, dtype=object)
    reason[species][~is_mapped[species]] = "unmapped"

    n_unmapped = int((~is_mapped[species]).sum())
    share = 100 * n_unmapped / len(meta)
    print(f"{species}: {n_unmapped:,} of {len(meta):,} nuclei have no position ({share:.1f}%)")

## 2. Degenerate

Positions are compared within each 10x lane. A position shared exactly by more than `max_at_coord` nuclei is blanked for all of them, as is one whose mean distance to its 5 nearest positions in the lane is below `floor_um` (mouse 10 µm, closer than nuclei can sit; human: not applied).

In [ ]:
def crowding_score(meta: pd.DataFrame, is_mapped: np.ndarray) -> pd.Series:
    """Mean distance (µm) to the K nearest positions of the same lane; NaN without a position."""
    score = pd.Series(np.nan, index=meta.index)
    for lane in meta.loc[is_mapped, SAMPLE].astype(str).unique():
        in_lane = is_mapped & (meta[SAMPLE].astype(str) == lane).to_numpy()
        score.loc[in_lane] = knn_mean_dist(meta.loc[in_lane, [X, Y]].to_numpy(float))
    return score


def nuclei_at_coordinate(meta: pd.DataFrame, is_mapped: np.ndarray) -> pd.Series:
    """Positioned nuclei of the same lane at exactly this coordinate; 0 without a position."""
    key = meta[SAMPLE].astype(str) + "|" + meta[X].astype(str) + "|" + meta[Y].astype(str)
    return key.map(key[is_mapped].value_counts()).fillna(0)


crowding = {}
for species in SPECIES:
    meta, settings = obs[species], SETTINGS[species]
    crowding[species] = crowding_score(meta, is_mapped[species])

    n_at_coordinate = nuclei_at_coordinate(meta, is_mapped[species])
    is_duplicate = is_mapped[species] & (n_at_coordinate > settings["max_at_coord"]).to_numpy()
    is_crowded = is_mapped[species] & (crowding[species] < settings["floor_um"]).to_numpy()
    reason[species][is_duplicate | is_crowded] = "degenerate"

    # A pile-up of exact duplicates draws as one dot on a map, so its size is reported here.
    pileups = (
        meta.loc[is_mapped[species]]
        .groupby([SAMPLE, X, Y], observed=True)
        .size()
        .sort_values(ascending=False)
    )
    lane, x_um, y_um = pileups.index[0]
    print(
        f"{species}: {int((is_duplicate | is_crowded).sum()):,} degenerate positions; "
        f"{int((pileups > settings['max_at_coord']).sum())} coordinates hold more than one nucleus, "
        f"the largest {pileups.iloc[0]:,} (lane {lane}, x = {x_um:.0f} µm, y = {y_um:.0f} µm)"
    )
    display(
        counts_by_timepoint(
            species,
            {
                "exact duplicate": is_duplicate,
                f"below {settings['floor_um']:g} µm": is_crowded,
                "degenerate": is_duplicate | is_crowded,
            },
        )
    )

## 3. Far

Blanking the degenerate positions changes everyone's neighbors, so the distance is recomputed on the positions that remain, per timepoint with the lanes of its puck pooled. The cut, median + `n_mads` MADs of log(1 + distance), is set per timepoint because density differs between sections.

In [ ]:
distance = {}
cut_um = {}
for species in SPECIES:
    meta = obs[species]
    survives = is_mapped[species] & (reason[species] != "degenerate").to_numpy()
    distance[species] = pd.Series(np.nan, index=meta.index)
    cut_um[species] = {}

    for timepoint in meta.loc[survives, TIMEPOINT].astype(str).unique():
        in_group = survives & (meta[TIMEPOINT].astype(str) == timepoint).to_numpy()
        dist_um = knn_mean_dist(meta.loc[in_group, [X, Y]].to_numpy(float))
        distance[species].loc[in_group] = dist_um
        cut_um[species][timepoint] = far_cut(dist_um, SETTINGS[species]["n_mads"])
        is_beyond = (distance[species] > cut_um[species][timepoint]).to_numpy()
        reason[species][in_group & is_beyond] = "far"

    is_far = (reason[species] == "far").to_numpy()
    is_scored = distance[species].notna().to_numpy()
    table = counts_by_timepoint(species, {"scored": is_scored, "far": is_far})
    table.insert(1, "cut (µm)", [round(cut_um[species][tp], 1) for tp in TIMEPOINT_ORDER[species]])
    print(f"{species}: {int(is_far.sum()):,} far positions")
    display(table)

## 4. Boxed

Only positions that passed both distance criteria can be boxed. A nucleus inside two regions counts in both rows of the table.

In [ ]:
BOUNDS = {
    "x_gt": (X, operator.gt),
    "x_lt": (X, operator.lt),
    "y_gt": (Y, operator.gt),
    "y_lt": (Y, operator.lt),
}


def in_region(meta: pd.DataFrame, region: dict) -> np.ndarray:
    """Positions of the region's timepoint that satisfy all of its bounds."""
    inside = (meta[TIMEPOINT].astype(str) == region["timepoint"]).to_numpy()
    for bound, (column, compare) in BOUNDS.items():
        if bound in region:
            inside &= compare(meta[column], region[bound]).to_numpy()
    return inside


for species in SPECIES:
    meta = obs[species]
    is_kept = (reason[species] == "kept").to_numpy()
    is_boxed = np.zeros(len(meta), bool)
    rows = []
    for region in SETTINGS[species]["exclusions"]:
        inside = is_kept & in_region(meta, region)
        is_boxed |= inside
        bounds = ", ".join(
            f"{bound[0]} {'>' if bound.endswith('gt') else '<'} {region[bound]:g}"
            for bound in BOUNDS
            if bound in region
        )
        rows.append({"timepoint": region["timepoint"], "bounds (µm)": bounds, "boxed": int(inside.sum())})
    reason[species][is_boxed] = "boxed"

    print(f"{species}: {int(is_boxed.sum()):,} boxed positions in {len(rows)} regions")
    if rows:
        display(pd.DataFrame(rows))

## Outcome per timepoint

The counts written to `spatial_qc_{species}.csv`.

In [ ]:
tally = {}
for species in SPECIES:
    meta = obs[species]
    # Rows in crosstab's alphabetical order, as in the tables of record.
    tally[species] = pd.crosstab(meta[TIMEPOINT].astype(str), reason[species]).reindex(
        columns=ORDER,
        fill_value=0,
    )
    tally[species].loc["__total__"] = tally[species].sum(axis=0)

    n_kept = int(tally[species].loc["__total__", "kept"])
    n_mapped = int(is_mapped[species].sum())
    print(f"{species}: {n_kept:,} of {n_mapped:,} positions kept ({100 * n_kept / n_mapped:.2f}%)")
    display(tally[species].rename_axis(columns=None))

## Sections and rotation

Each kept nucleus gets the tissue section it lies in: `{lane}__A`, or `{lane}__A`/`{lane}__B` on either side of a split line. Blanked nuclei get none. A section under 200 nuclei would suggest a misplaced line. The rotated positions equal the QC'd ones while every angle is 0.

In [ ]:
def assign_sections(meta: pd.DataFrame, is_kept: np.ndarray, lines: list[dict]) -> pd.Series:
    """Tissue section of each kept nucleus: its lane, plus the side of a split line."""
    section = pd.Series(pd.NA, index=meta.index, dtype=object)
    section[is_kept] = meta.loc[is_kept, SAMPLE].astype(str) + "__A"
    for split in lines:
        line = split["line"]
        on_puck = is_kept & (meta[TIMEPOINT].astype(str) == split["timepoint"]).to_numpy()
        y_on_line = line["slope"] * meta[X].astype(float) + line["intercept"]
        is_above = (meta[Y].astype(float) > y_on_line).to_numpy()
        for side, label in ((on_puck & is_above, split["above"]), (on_puck & ~is_above, split["below"])):
            section[side] = meta.loc[side, SAMPLE].astype(str) + "__" + label
    return section


def rotate_positions(
    meta: pd.DataFrame,
    is_mapped: np.ndarray,
    is_kept: np.ndarray,
    x_qc: np.ndarray,
    y_qc: np.ndarray,
    angles: dict[str, float],
) -> tuple[np.ndarray, np.ndarray]:
    """QC'd positions rotated per lane by their timepoint's angle (degrees, counterclockwise)."""
    x_rot, y_rot = x_qc.copy(), y_qc.copy()
    for lane in meta.loc[is_mapped, SAMPLE].astype(str).unique():
        in_lane = is_mapped & (meta[SAMPLE].astype(str) == lane).to_numpy()
        angle = float(angles.get(meta.loc[in_lane, TIMEPOINT].astype(str).iloc[0], 0.0))
        if angle % 360 == 0:
            continue

        # The center is the lane's median kept position; a lane with few kept positions uses all.
        center_from = in_lane & is_kept if (in_lane & is_kept).sum() >= 20 else in_lane
        cx = float(np.nanmedian(meta.loc[center_from, X]))
        cy = float(np.nanmedian(meta.loc[center_from, Y]))
        theta = np.radians(angle)
        dx, dy = x_qc[in_lane] - cx, y_qc[in_lane] - cy
        x_rot[in_lane] = cx + np.cos(theta) * dx - np.sin(theta) * dy
        y_rot[in_lane] = cy + np.sin(theta) * dx + np.cos(theta) * dy
    return x_rot, y_rot


qc = {}
for species in SPECIES:
    meta, settings = obs[species], SETTINGS[species]
    is_kept = (reason[species] == "kept").to_numpy()
    x_qc = np.where(is_kept, meta[X].astype(float), np.nan)
    y_qc = np.where(is_kept, meta[Y].astype(float), np.nan)

    section = assign_sections(meta, is_kept, settings["sections"])
    no_angle = sorted(set(meta.loc[is_mapped[species], TIMEPOINT].astype(str)) - set(settings["rotations"]))
    if no_angle:
        print(f"{species}: no rotation angle for {no_angle}; they stay unrotated")
    x_rot, y_rot = rotate_positions(meta, is_mapped[species], is_kept, x_qc, y_qc, settings["rotations"])

    qc[species] = pd.DataFrame(
        {
            "spatial_qc_reason": pd.Categorical(reason[species].to_numpy(), categories=ORDER),
            "x_um_qc": x_qc,
            "y_um_qc": y_qc,
            "x_um_rot": x_rot,
            "y_um_rot": y_rot,
            "tissue_section": pd.Categorical(section.astype(object).where(section.notna(), None)),
        },
        index=meta.index,
    )

    section_sizes = qc[species]["tissue_section"].value_counts().sort_index()
    small = section_sizes[section_sizes < 200]
    if len(small):
        print(f"{species}: sections under 200 nuclei, check the split line: {small.to_dict()}")
    print(f"{species}: {len(section_sizes)} sections")
    display(section_sizes.rename("kept nuclei").to_frame())

## Maps

Raw positions of every timepoint, colored by outcome, with the exclusion regions (dashed) and the uninduced split line (dotted). Blanked positions are drawn on top of the kept ones and larger, so that the few hundred of them stay visible among thousands of kept positions.

In [ ]:
panels = [(species, tp) for species in SPECIES for tp in TIMEPOINT_ORDER[species]]
n_cols = 4
n_rows = int(np.ceil(len(panels) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(11, 2.9 * n_rows), layout="constrained", squeeze=False)

for ax, (species, timepoint) in zip(axes.flat, panels):
    meta = obs[species]
    in_panel = is_mapped[species] & (meta[TIMEPOINT].astype(str) == timepoint).to_numpy()

    # Kept positions first and small; blanked ones on top, by outcome.
    for outcome, size in (("kept", 0.3), ("degenerate", 4), ("far", 4), ("boxed", 4)):
        drawn = in_panel & (reason[species] == outcome).to_numpy()
        ax.scatter(
            meta.loc[drawn, X],
            meta.loc[drawn, Y],
            s=size,
            color=OUTCOME_COLORS[outcome],
            linewidths=0,
        )

    # Equal axes in µm, so the regions read in the units of their bounds.
    ax.set_aspect("equal")
    ax.autoscale_view()
    xlim, ylim = ax.get_xlim(), ax.get_ylim()

    # Regions are half-planes or corners; open sides end at the panel edge.
    for region in SETTINGS[species]["exclusions"]:
        if region["timepoint"] == timepoint:
            x0, x1 = region.get("x_gt", xlim[0]), region.get("x_lt", xlim[1])
            y0, y1 = region.get("y_gt", ylim[0]), region.get("y_lt", ylim[1])
            ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ls="--", lw=0.8, ec=REGION_COLOR))
    for split in SETTINGS[species]["sections"]:
        if split["timepoint"] == timepoint:
            line = split["line"]
            xs = np.array(xlim)
            ax.plot(xs, line["slope"] * xs + line["intercept"], ls=":", lw=0.8, color="black")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    n_blanked = int((in_panel & ~(reason[species] == "kept").to_numpy()).sum())
    ax.set_title(f"{species.capitalize()}, {TIMEPOINT_LABELS[timepoint]}: {n_blanked:,} blanked")
    ax.tick_params(labelsize=6)

for ax in axes.flat[len(panels):]:
    ax.set_visible(False)

handles = [Line2D([], [], ls="", marker="o", color=OUTCOME_COLORS[o], label=o) for o in OUTCOME_COLORS]
handles.append(Line2D([], [], ls="--", color=REGION_COLOR, label="exclusion region"))
fig.legend(handles=handles, loc="outside upper center", ncols=len(handles), frameon=False)
fig.supxlabel("x (µm)")
fig.supylabel("y (µm)")
plt.show()

## Human: what the mouse criteria would remove

Not applied. The mouse crowding floor, then the mouse far cut on the positions that remain, per human timepoint.

In [ ]:
mouse_settings = SETTINGS["mouse"]
meta = obs["human"]
not_degenerate = is_mapped["human"] & (reason["human"] != "degenerate").to_numpy()
would_be_crowded = not_degenerate & (crowding["human"] < mouse_settings["floor_um"]).to_numpy()
would_survive = not_degenerate & ~would_be_crowded

whatif_distance = pd.Series(np.nan, index=meta.index)
whatif_cut_um = {}
rows = []
for timepoint in TIMEPOINT_ORDER["human"]:
    in_timepoint = (meta[TIMEPOINT].astype(str) == timepoint).to_numpy()
    in_group = would_survive & in_timepoint
    dist_um = knn_mean_dist(meta.loc[in_group, [X, Y]].to_numpy(float))
    whatif_distance.loc[in_group] = dist_um
    whatif_cut_um[timepoint] = far_cut(dist_um, mouse_settings["n_mads"])
    rows.append(
        {
            "timepoint": TIMEPOINT_LABELS[timepoint],
            f"below {mouse_settings['floor_um']:g} µm": int((would_be_crowded & in_timepoint).sum()),
            "far cut (µm)": round(whatif_cut_um[timepoint], 1),
            "far": int((dist_um > whatif_cut_um[timepoint]).sum()),
        }
    )
whatif = pd.DataFrame(rows).set_index("timepoint")
print(
    f"human: the mouse floor would blank {whatif.iloc[:, 0].sum():,} more positions "
    f"and the mouse far cut {whatif['far'].sum():,} more"
)
display(whatif)

## Far-step distances

Distribution of the far-step distance per timepoint, with the cut (dashed). The axes are logarithmic because distances span two orders of magnitude and the far tail holds a few dozen nuclei per timepoint. Human panels show the mouse criteria of the previous section, which are not applied.

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(11, 2.3 * n_rows), layout="constrained", squeeze=False)

# One set of bins for all panels: 0.05 log10 units (12% steps in distance).
all_dist = pd.concat([distance["mouse"], whatif_distance]).dropna()
all_dist = all_dist[all_dist > 0]
bins = 10 ** np.arange(np.floor(np.log10(all_dist.min()) * 20) / 20, np.log10(all_dist.max()) + 0.05, 0.05)

for ax, (species, timepoint) in zip(axes.flat, panels):
    meta = obs[species]
    in_timepoint = (meta[TIMEPOINT].astype(str) == timepoint).to_numpy()
    if species == "human":
        dist_um = whatif_distance[in_timepoint].dropna()
        cut, note = whatif_cut_um[timepoint], " (not applied)"
    else:
        dist_um = distance[species][in_timepoint].dropna()
        cut, note = cut_um[species][timepoint], ""

    ax.hist(dist_um, bins=bins, histtype="step", color="dimgrey")
    ax.axvline(cut, ls="--", lw=0.8, color=CUT_COLOR)
    ax.set_xscale("log")
    ax.set_yscale("log")
    n_far = int((dist_um > cut).sum())
    ax.set_title(f"{species.capitalize()}, {TIMEPOINT_LABELS[timepoint]}\n{n_far} far (> {cut:.0f} µm){note}")

for ax in axes.flat[len(panels):]:
    ax.set_visible(False)
fig.supxlabel(f"Mean distance to the {K} nearest positions (µm)")
fig.supylabel("Nuclei per bin")
plt.show()

## Write

The output object is the input with the QC columns added; the raw positions are not touched.

In [ ]:
def qc_object(
    adata: ad.AnnData,
    qc_columns: pd.DataFrame,
    settings: dict,
    cuts: dict[str, float],
) -> ad.AnnData:
    """Copy of the input with the QC columns, QC'd and rotated obsm positions, and the settings in uns."""
    adata = adata.copy()
    for column in qc_columns:
        adata.obs[column] = qc_columns[column]
    adata.obsm["spatial"] = np.c_[qc_columns["x_um_qc"], qc_columns["y_um_qc"]]
    adata.obsm["spatial_rot"] = np.c_[qc_columns["x_um_rot"], qc_columns["y_um_rot"]]
    adata.uns["spatial_qc"] = {
        "k": K,
        "floor_um": settings["floor_um"],
        "max_at_coord": settings["max_at_coord"],
        "n_mads": settings["n_mads"],
        "mad_scale": MAD_SCALE,
        "mad_group": TIMEPOINT,
        "log_scale": True,
        "action": "blank",
        "drop_unmapped": False,
        "x_key": X,
        "y_key": Y,
        "thresholds_um": {tp: round(cut, 1) for tp, cut in cuts.items()},
        "exclusions_json": json.dumps(settings["exclusions"]),
        "rotations_json": json.dumps({tp: float(angle) for tp, angle in settings["rotations"].items()}),
        "sections_json": json.dumps(settings["sections"], default=str),
    }
    return adata


qc_adatas = {
    species: qc_object(adatas[species], qc[species], SETTINGS[species], cut_um[species])
    for species in SPECIES
}

if OUT_DIR is None:
    print("OUT_DIR is None: nothing written")
else:
    out_dir = resolve(OUT_DIR)
    targets = {
        species: (out_dir / f"qc_spatial_{species}.h5ad", out_dir / f"spatial_qc_{species}.csv")
        for species in SPECIES
    }
    existing = [str(path) for paths in targets.values() for path in paths if path.exists()]
    if existing and not FORCE:
        raise FileExistsError(f"Refusing to overwrite {existing}; set FORCE = True")

    out_dir.mkdir(parents=True, exist_ok=True)
    for species, (h5ad_path, tally_path) in targets.items():
        qc_adatas[species].write_h5ad(h5ad_path)
        tally[species].to_csv(tally_path)
        print(f"wrote {h5ad_path}\nwrote {tally_path}")

## Checks

Every count is the number of nuclei (or settings) that differ; all must be 0. The release objects hold the subset of nuclei kept by annotation, with the labels and QC'd positions of the QC of record.

In [ ]:
def n_differ(a, b) -> int:
    """Number of positions where two equally indexed vectors differ; NaN equals NaN."""
    a, b = np.asarray(a), np.asarray(b)
    if a.dtype.kind == "f" or b.dtype.kind == "f":
        return int((~np.isclose(a.astype(float), b.astype(float), rtol=0, atol=0, equal_nan=True)).sum())
    return int((a.astype(str) != b.astype(str)).sum())


def plain(value):
    """uns values as plain Python, so that h5ad round trips compare equal."""
    if isinstance(value, dict):
        return {key: plain(item) for key, item in value.items()}
    return value.item() if hasattr(value, "item") else value


def read_obs_columns(path: Path, columns: list[str]) -> pd.DataFrame:
    """Selected obs columns of an h5ad, without reading the expression matrix."""
    with h5py.File(path, "r") as f:
        group = f["obs"]
        index = np.asarray(read_elem(group[group.attrs["_index"]])).astype(str)
        return pd.DataFrame({col: read_elem(group[col]) for col in columns}, index=index)


QC_COLUMNS = ["spatial_qc_reason", "x_um_qc", "y_um_qc", "x_um_rot", "y_um_rot", "tissue_section"]
checks = []
for species in SPECIES:
    if RECORD.get(species):
        record = ad.read_h5ad(resolve(RECORD[species]))
        mine = qc_adatas[species]
        if not record.obs.index.equals(mine.obs.index):
            raise ValueError(f"{species}: the record holds other nuclei than the input")

        for column in QC_COLUMNS:
            checks.append((species, "record", column, n_differ(mine.obs[column], record.obs[column])))
        for key in ("spatial", "spatial_rot"):
            checks.append((species, "record", f"obsm {key}", n_differ(mine.obsm[key], record.obsm[key])))
        mine_uns, record_uns = plain(mine.uns["spatial_qc"]), plain(record.uns["spatial_qc"])
        all_keys = mine_uns.keys() | record_uns.keys()
        differing = [key for key in all_keys if mine_uns.get(key) != record_uns.get(key)]
        checks.append((species, "record", "uns spatial_qc", len(differing)))

    if RECORD_TALLY.get(species):
        same_text = tally[species].to_csv() == resolve(RECORD_TALLY[species]).read_text()
        checks.append((species, "record", "tally CSV", int(not same_text)))

    if RELEASE.get(species):
        release_columns = ["spatial_qc_reason", X, Y, "x_um_qc", "y_um_qc"]
        released = read_obs_columns(resolve(RELEASE[species]), release_columns)
        absent = released.index.difference(qc[species].index)
        if len(absent):
            raise ValueError(f"{species}: {len(absent)} released nuclei are not in the input")

        mine = pd.concat([obs[species][[X, Y]], qc[species]], axis=1).loc[released.index]
        for column in released.columns:
            against = f"release ({len(released):,} nuclei)"
            checks.append((species, against, column, n_differ(mine[column], released[column])))

checks = pd.DataFrame(checks, columns=["species", "against", "compared", "n differ"])
display(checks)
if checks["n differ"].any():
    raise AssertionError("the QC does not reproduce the results of record; see the table above")
print(f"All {len(checks)} checks pass.")